In [2]:
# ── live_tracker.py real-DB smoke test ──
# Paste your actual DB credentials below
import os

MONGO_URI = os.getenv("MONGO_URI", "")
#DB_URL = "postgresql+psycopg2://hbot:<YOUR_POSTGRES_PASSWORD>@<YOUR_TRUENAS_IP>:5432/hummingbot_api"
DB_URL = MONGO_URI


import sys
sys.path.insert(0, "research_notebooks/market_lab/pmm_dynamic")

from pmm_lab.deploy.live_tracker import LivePerformanceTracker, TrackerHealth

tracker = LivePerformanceTracker(DB_URL)

# 1. Can we reach the DB?
print(f"Ping: {tracker.ping()}")

# 2. Schema check happens automatically on first get_trades() call.
#    We'll see the result in tracker.last_health after the query.

# 3. Try fetching trades
CONNECTOR = "nonkyc"
PAIR = "XMR-USDT"  # <-- change to a pair you've traded
HOURS = 168

trades = tracker.get_trades(CONNECTOR, PAIR, hours=HOURS)
print(f"\nHealth after query: {tracker.last_health}")
if tracker.last_health == TrackerHealth.SCHEMA_ERROR:
    print(f"  SCHEMA ERROR: {tracker.last_error}")
print(f"Trades found: {len(trades)}")

if trades:
    t = trades[0]
    print(f"\nFirst trade sample:")
    print(f"  trade_id:     {t.trade_id}")
    print(f"  pair:         {t.trading_pair}")
    print(f"  side:         {t.side}")
    print(f"  price:        {t.price}")
    print(f"  amount:       {t.amount}")
    print(f"  fee_amount:   {t.fee_amount}")
    print(f"  fee_currency: {t.fee_currency}")
    print(f"  fee_source:   {t.fee_source}")
    print(f"  timestamp:    {t.timestamp}")
    print(f"  order_type:   {t.order_type}")
    
    # 4. Full metrics
    metrics = tracker.get_performance(CONNECTOR, PAIR, hours=HOURS)
    print(f"\nPerformance ({HOURS}h):")
    print(f"  Trades: {metrics.trade_count} ({metrics.buy_count}B / {metrics.sell_count}S)")
    print(f"  Volume: {metrics.total_volume_base:.6f} base / {metrics.total_volume_quote:.2f} quote")
    print(f"  Fees:   {metrics.total_fees_quote:.6f} quote")
    print(f"  Avg buy:  {metrics.avg_buy_price:.8f}")
    print(f"  Avg sell: {metrics.avg_sell_price:.8f}")
    print(f"  Est PnL:  {metrics.estimated_pnl_quote:.6f} quote")
    if metrics.unresolved_fee_count > 0:
        print(f"  ⚠ Unresolved fees: {metrics.unresolved_fee_count} in {metrics.unresolved_fee_currencies}")
    print(f"\n  _trades populated: {metrics._trades is not None and len(metrics._trades) > 0}")
else:
    print(f"\nNo trades found for {CONNECTOR}/{PAIR} in {HOURS}h")
    if tracker.last_error:
        print(f"  Error: {tracker.last_error}")

# 5. Column inspection
print("\n── TradeFill columns in your DB ──")
from sqlalchemy import create_engine, text
engine = create_engine(DB_URL)
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT column_name FROM information_schema.columns WHERE table_name = 'TradeFill'"
    ))
    for row in result:
        print(f"  {row[0]}")

# 6. What connectors exist?
print("\n── Distinct connectors (market column) ──")
with engine.connect() as conn:
    result = conn.execute(text('SELECT DISTINCT market FROM "TradeFill"'))
    for row in result:
        print(f"  {row[0]}")

Ping: False


NoSuchModuleError: Can't load plugin: sqlalchemy.dialects:mongodb